Sample the SPE data

In [3]:
# 1. FORCE JUPYTER TO RELOAD CHANGED .py FILES
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.gee_sampler import HydroSampler

# ==========================================
# 1. CONFIGURATION
# ==========================================
GAUGE_CSV   = PROJECT_ROOT / "data" / "gauges" / "sarawak_all_gauges.csv"
OUTPUT_FILE = PROJECT_ROOT / "outputs" / "processed" / "sarawak_hydro_data_FULL.parquet"
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. INITIALIZE & RUN
# ==========================================
sampler = HydroSampler(
    gauge_csv=str(GAUGE_CSV),
    col_station="station_id",  # <--- CRITICAL FIX: Use Station ID, not Name
    col_lat="latitude",
    col_lon="longitude",
    start_date="1983-01-01",
    end_date="2024-12-31",
    project_id="ee-azaleakamellia",
    sampling="native",      
    master_scale=5566,
)

df = sampler.run_pipeline(
    spe_list=["CHIRPS", "ERA5_LAND", "PERSIANN"],
    pet_product="ERA5_LAND",
    dem_product="SRTM",
    lulc_product="MODIS",
    output_path=str(OUTPUT_FILE),
)

print("\n--- First 5 Rows ---")
print(df.head())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ GEE initialized: ee-azaleakamellia

🚀 HydroSampler | 1983-01-01 → 2024-12-31 | sampling=native
------------------------------------------------------------
  → precip_chirps | UCSB-CHG/CHIRPS/DAILY | scale=5566 m
      1983: 335 stations, 4020 rows, 4008 valid
      1984: 335 stations, 4020 rows, 4008 valid
      1985: 335 stations, 4020 rows, 4008 valid
      1986: 335 stations, 4020 rows, 4008 valid
      1987: 335 stations, 4020 rows, 4008 valid
      1988: 335 stations, 4020 rows, 4008 valid
      1989: 335 stations, 4020 rows, 4008 valid
      1990: 335 stations, 4020 rows, 4008 valid
      1991: 335 stations, 4020 rows, 4008 valid
      1992: 335 stations, 4020 rows, 4008 valid
      1993: 335 stations, 4020 rows, 4008 valid
      1994: 335 stations, 4020 rows, 4008 valid
      1995: 335 stations, 4020 rows, 4008 valid
      1996: 335 stations, 4020 rows, 4008 valid
      1997: 335 stations,

Converting Excel of rainfall stations to the correct CSV.

In [2]:
import pandas as pd
import re

def dms_to_dd(dms_str):
    """Converts DMS string (e.g., '1º 36\' 0.10" N') to Decimal Degrees."""
    if pd.isna(dms_str): return None
    match = re.search(r"(\d+)º\s*(\d+)'\s*([\d.]+)\"\s*([NSEW])", str(dms_str))
    if match:
        deg, mins, secs, direction = float(match.group(1)), float(match.group(2)), float(match.group(3)), match.group(4)
        dd = deg + (mins / 60) + (secs / 3600)
        return round(-dd if direction in ['S', 'W'] else dd, 6)
    return None

# 1. Load the Excel file
excel_path = r"C:\Users\User\Downloads\DID_Sarawak_Rainfall_Station_2025.xlsx"
df = pd.read_excel(excel_path)

# 2. Identify columns dynamically to avoid header issues
cols = df.columns
id_col = cols[0]       # First column is Station ID
name_col = cols[1]     # Second column is Station Name
lat_col = next(c for c in cols if 'LAT' in str(c).upper())
lon_col = next(c for c in cols if 'LON' in str(c).upper())

# 3. Extract and clean
out_df = pd.DataFrame({
    'station_id': df[id_col].astype(str).str.strip(),
    'station_name': df[name_col].astype(str).str.strip(),
    'latitude': df[lat_col].apply(dms_to_dd),
    'longitude': df[lon_col].apply(dms_to_dd)
})

# 4. Drop rows with missing coordinates and remove duplicates based on station_id
out_df = out_df.dropna(subset=['latitude', 'longitude'])
out_df = out_df.drop_duplicates(subset=['station_id'])

# 5. Save to CSV
output_csv = r"D:\my_project\data\gauges\sarawak_all_gauges.csv"
out_df.to_csv(output_csv, index=False)
print(f"✅ Successfully saved {len(out_df)} unique stations to: {output_csv}")

✅ Successfully saved 335 unique stations to: D:\my_project\data\gauges\sarawak_all_gauges.csv
